
### Install Duck Duck Go and Trafilatura (scrapping tool) libraries
##### (another scrapping tool is 'Beautiful Soup')

#### also AgentAI is also known as ReAct (Reasoning and acting) Refer the link below
https://www.ibm.com/think/topics/react-agent

In [37]:
# one time installation; commented after installation
# !pip install ddgs trafilatura

### step1: Setup

In [55]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
import trafilatura


load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])

client = OpenAI()

MODEL = "gpt-4.1-mini"

sk-proj-


### Step 2 : Define the tools

In [39]:
def search_web(query: str):
    """ Search the web using DubkDuckGo browser. Reeturn 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f" \u2705 Got results")
    return json.dumps(results, indent=2) # this step is to covert list into text as LLM underestands only text and not the list

In [40]:

def fetch_url(url:str):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 Got text: {len(text)} chars")
            return text
    print(f" \u274C failed to fech or extract text from url: {url}")
    return f" Could not extract text from {url}. Try a different source"




In [41]:


# test search_eb
message = "AI in healthcare in 2030"
searchRet = search_web(message)
searchRet


 ✅ Got results


'[\n  {\n    "title": "AI in Healthcare 2030: Opportunities, Challenges, and Strategic Insights",\n    "href": "https://www.linkedin.com/pulse/ai-healthcare-2030-opportunities-challenges-strategic-riken-shah-pk86f",\n    "body": "Strategic Insights: Preparing For 2030. By 2030, AI will be an active partner, learning, adapting, and collaborating with clinicians. To leverage its potential: Invest in clinician AI literacy \\u2013 Ensure staff understand AI\\u2019s capabilities, limitations, and ethical use."\n  },\n  {\n    "title": "Artificial Intelligence (AI) In Healthcare Market Growth... | Technavio",\n    "href": "https://www.technavio.com/report/artificial-intelligence-market-in-healthcare-sector-industry-analysis",\n    "body": "Ada Health GmbH - Delivers AI-powered digital health diagnostics, providing advanced symptom assessment and critical patient insights to enhance care delivery.What is the expected growth of the Artificial Intelligence (AI) In Healthcare Market between 2026

In [42]:
# test fetch url
url = "https://www.grandviewresearch.com/industry-analysis/artificial-intelligence-ai-healthcare-market"
fetch_url(url)

 ✅ Got text: 27811 chars


'- Home\n- »\n- Healthcare IT\n- »\n-\nAI In Healthcare Market Size & Share, Industry Report, 2033GVR Report cover\nArtificial Intelligence In Healthcare Market (2026 - 2033) Size, Share & Trends Analysis Report By Component (Hardware, Software, Services), By Application (Clinical Trials, Cybersecurity), By End-use, By Technology, By Region, And Segment Forecasts\n- Report ID: GVR-3-68038-951-7\n- Number of Report Pages: 150\n- Format: PDF\n- Historical Range: 2021 - 2025\n- Forecast Period: 2026 - 2033\n- Industry: Healthcare\n- Report Summary\n- Table of Contents\n- Segmentation\n- Methodology\n- Download FREE Sample\n- Download Sample Report\nAI In Healthcare Market Summary\nThe global artificial intelligence in healthcare market size was estimated at USD 36.67 billion in 2025 and is projected to reach USD 505.59 billion by 2033, growing at a CAGR of 38.90% from 2026 to 2033. The key factor driving market growth is the increasing demand in the healthcare sector for enhanced efficien

In [49]:



search_web_function = {

    "name" : "search_web",
    "description": "Search the web using DubkDuckGo browser. Reeturn 3 results",
    "parameters": {
        "type": "object",
        "properties": {

            "query": {
                "type": "string",
                "description": "The search query to find relevant websites"
            }
        },
        "required":["query"]
    }
}
tools = [{"type": "function", "function": search_web_function}]

fetch_url_function = {

    "name" : "fetch_url",
    "description": "Fetch and extract the main text content from a web page",
    "parameters": {
        "type": "object",
        "properties": {

            "url": {
                "type": "string",
                "description": "The url of the web page to fetch and extract text from"
            }
        },
        "required":["url"]
    }
}

tools.append({"type": "function", "function": fetch_url_function})

tools

[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'Search the web using DubkDuckGo browser. Reeturn 3 results',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query to find relevant websites'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'Fetch and extract the main text content from a web page',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'The url of the web page to fetch and extract text from'}},
    'required': ['url']}}}]

### Step3: Tool Call Hander

In [68]:
# handle tool call
def handle_tool_call(tool_calls):
    # .....
    # return what to add to our "coontext" about the tool call results, a dictionary

    tool_call_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f" \U0001F527 Calling function {function_name} with argument: {args}")
        if(function_name == "search_web"):
            
            #send notification
            result = search_web(args["query"])
            content = f"Search Web Results: {result}"
            print(content)
            
        elif (function_name == "fetch_url"):
            result = fetch_url(args["url"])
            content = f"Fetched URL content: {result}"
        else:
            content = f"Unknown function: {function_name}"
     
        tool_call_result = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": content
            }
        tool_call_results.append(tool_call_result)

    return tool_call_results

### Steep 4: The System Prompt
### This tells the LLM who it is and how to behave. The key things:

### 1) What it job is?
### 2) What tools it has?
### 3) What process to follow?
### 4) what output format to product?

In [ ]:
# My version ( not optimal)
RESEARCH_AGENT_PROMPT = """
You are a helpful research assistant ageent. Your job as an ageent is to find the relevant data to the user query.
The process is to scrape the data first from the web for which you will use the search_web tool which will give you results.
The result contains the href, which has the url link. extract the url whichh is in the HREF field and pass to another tool fetch_url to get the content.
Use that content to provide output in a markndown format listing your finding. Use based to summarize the data as best as possible and refer the url where possible. if any reason you cannot find the relevant informatoin then don't hesitate to call it out. 
"""

# kirill vesion 
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word 'DONE:' is a control signal and not a label. Only use the word "DONE:" as per the instruction below -- it has to come at the start of a message

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather informaton from at least 6 distinct sources before delviering your brief.
if you have a fewer than 6 sources then keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that DONE: should be at the start of the final response, so it can be easily extracted and parsed.
You CANNOT and SHOULD NOT include DONE: in any part of the response except at the very beginning of the final response.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step5 : The Agentic Loop

In [56]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
    """
        Run the research agent on a topic and return the research brief
        Args:
            topic: The topic to research
            max_iterations: safetly limit to prevent infinite loop
    
    """
    print(f"\n\u0001F50D Starting research on {topic}")

    # Initialize conversation messages list with system prompt + research talk
    messages = [{"role": "system", "content": RESEARCH_AGENT_PROMPT},
                {"role": "user", "content": f"Reserch the following topic and produce a comprehensive research brief:\n {topic}"}]
    #Loop
        #1. Call the LLM and get response
        #2. Chek if LLM called tools

        #3. Otherwise: no tools were called, read message content
            #Check if DONE:, then return
            #Otherwise: not yet done, append message
        #4 if we are entering the final iteration, force a final answer

    #fallback return
    iteration = 1
    while iteration <= max_iterations:
   
        response = client.chat.completions.create(
            model = MODEL,
            messages = messages,
            tools = tools
        )

        message = response.choices[0].message
        messages.append(message)
        print(f"*******iteration: {iteration}\n")
        if message.tool_calls:
            from pprint import pprint
            pprint(message.tool_calls)

            tool_call_results = handle_tool_call(message.tool_calls)
       
            messages.extend(tool_call_results)
        else: # no tool calls, so there just a normal text response from the LLM
            content = message.content
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"\n\u2705 Research complete!")
                return research_brief
            else:
                print(f"  \u001F4AD Agent is thinking:")
                pprint(content)
                #loop continues to next iteration
        
        if(iteration == max_iterations -1):
            print("  \u26a0 Safety limit reachd. Stopping research in next iteeration")
            messages.append({"role": "user", "content": "You have reached a maximum number of iterations, Please deliver your response. Please deliver your response with DONE: followed by breif"})
    
        iteration+=1 # increment the count
    return "Research incomplete! Max iterations reached without finalzing the brief"

### Lets Run it

In [73]:
# brief = run_research_agent("AI in healthcare in 2030")
brief = run_research_agent("Prediction of the IPL 2026 winner based on the current status")
display(Markdown(brief))


F50D Starting research on Prediction of the IPL 2026 winner based on the current status
*******iteration: 1

[ChatCompletionMessageFunctionToolCall(id='call_IV3V66z7XCYtgo3YGiqMkHxF', function=Function(arguments='{"query":"IPL 2026 winner prediction based on current team status"}', name='search_web'), type='function')]
 🔧 Calling function search_web with argument: {'query': 'IPL 2026 winner prediction based on current team status'}
 ✅ Got results
Search Web Results: [
  {
    "title": "IPL 2026 Winner Prediction - Who Will Win? | IPL Predict",
    "href": "https://iplpredict.in/ipl-winner-prediction-2026/",
    "body": "Our IPL 2026 winner prediction breaks down all 10 teams, identifies the top contenders, and assigns win probabilities based on historical data and current squad strength."
  },
  {
    "title": "IPL 2026 Winner Prediction: Which Team Will Lift the Trophy?",
    "href": "https://allcric.com/blog/ipl-2026-winner-prediction/",
    "body": "Who will win IPL 2026? Explore 

The prediction for IPL 2026 winner, based on current team status and mid-season data, highlights a competitive tournament shaped by form, squad depth, historical trends, and player fitness.

Key facts and statistics:
- IPL 2026 runs Mar 28 – May 31, with 10 teams and 74-84 matches.
- Punjab Kings (PBKS) are mid-season leaders with 6-0-1 record, undefeated so far, and highest win probability (~23%).
- Defending champions Royal Challengers Bengaluru (RCB) have a 16-22% win probability and stable core squad.
- Mumbai Indians (MI) historically strong with 5 titles, have key bowlers (Bumrah, Boult) and deep batting but struggled mid-season.
- Chennai Super Kings (CSK), also 5-time champions, are recovering after a slow start and key acquisition of Sanju Samson.
- Teams like Sunrisers Hyderabad (SRH) have improved with Pat Cummins’ return, making them a value pick.
- Kolkata Knight Riders (KKR) and Gujarat Titans (GT) face injury and form issues, lowering their chances.

Main themes:
- Top contenders include PBKS (current form), MI (depth and bowling), RCB (defending champs), and CSK (experience and new signings).
- Historical data shows back-to-back titles are rare; defending champs face strong pressure.
- Successful teams generally have a strong Indian core, effective death bowling, and coaching stability.
- Injuries, overseas player availability, and early match momentum significantly influence outcomes.
- Betting markets and predictive models largely align on PBKS as favorites but note slightly inflated odds; value exists in SRH and some others.

Notable data points:
- PBKS undefeated mid-season at 6-0-1 with Net Run Rate +1.333.
- MI’s elite pace duo of Bumrah and Boult is key to their chances.
- RCB retained 17 of 18 players from 2025 title squad.
- CSK’s Sanju Samson was the T20 World Cup 2026 Player of the Tournament.
- Pre-tournament favorites like KKR and GT lost value due to injuries and inconsistent form.

Sources:
- https://iplpredict.in/ipl-winner-prediction-2026/
- https://allcric.com/blog/ipl-2026-winner-prediction/
- https://cricketprediction.com/ipl/winner-prediction-2026

### Assignment 1: Revision : Concise Research Agent

In [62]:
RESEARCH_AGENT_PROMPT = """
You are a helpful research assistant ageent. Your job as an ageent is to find the relevant data to the user query.
The process is to scrape the data first from the web for which you will use the search_web tool which will give you results.
The result contains the href, which has the url link. extract the url whichh is in the HREF field and pass to another tool fetch_url to get the content.
Use that content to provide output in a markndown format listing your finding. Use based to summarize the data as best as possible and refer the url where possible. if any reason you cannot find the relevant informatoin then don't hesitate to call it out. 
"""

# kirill vesion 
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word 'DONE:' is a control signal and not a label. Only use the word "DONE:" as per the instruction below -- it has to come at the start of a message

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that DONE: should be at the start of the final response, so it can be easily extracted and parsed.
You CANNOT and SHOULD NOT include DONE: in any part of the response except at the very beginning of the final response.

Your research brief MUST include following informatoin. However this brief should be very concice and short and not too wordy or lengthy:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Summarize your response to be brief and concise and too wordy or lengthy

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

In [63]:
# MODEL = ""
brief = run_research_agent("Prediction of the IPL 2026 winner based on the current status")
display(Markdown(brief))


F50D Starting research on Prediction of the IPL 2026 winner based on the current status
*******iteration: 1

[ChatCompletionMessageFunctionToolCall(id='call_SaBzXwb15PvaVmb3Cmgmw5Yr', function=Function(arguments='{"query":"IPL 2026 winner prediction based on current team status"}', name='search_web'), type='function')]
 🔧 Calling function search_web with argument: {'query': 'IPL 2026 winner prediction based on current team status'}
 ✅ Got results
Search Web Results: [
  {
    "title": "2026 Indian Premier League - Wikipedia",
    "href": "https://en.wikipedia.org/wiki/2026_Indian_Premier_League",
    "body": "The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, is the 19th edition of the Indian Premier League, a professional Twenty20 cricket league."
  },
  {
    "title": "IPL 2026 Fixtures, Live Score, Schedule, Points Table & News",
    "href": "https://www.espncricinfo.com/series/ipl-2026-1510719",
    "body": "Get full coverage of IPL 2026 with live 

Research Brief: Prediction of the IPL 2026 Winner Based on Current Status

Key Facts & Statistics:
- IPL 2026 (IPL 19) runs from 28 March to 31 May with 10 teams playing 74 matches.  
- Royal Challengers Bengaluru (RCB) are the defending champions (won their first title in 2025).  
- Current points table top contenders (end of group stage):  
  - Punjab Kings: 6W, 1L, 1NR, 13 points (1st position)  
  - Royal Challengers Bengaluru: 6W, 2L, 12 points (2nd)  
  - Rajasthan Royals: 6W, 3L, 12 points (3rd)  
  - Sunrisers Hyderabad and Gujarat Titans trail behind with 10 and 8 points respectively.  
- Player highlights: Vaibhav Sooryavanshi (Rajasthan Royals) leads runs (400), Abhishek Sharma (Sunrisers Hyderabad) close behind (380). Kagiso Rabada (Rajasthan Royals) and others ranked high for points and wickets.  
- Key players: RCB with Rajat Patidar leading, Punjab Kings with Shreyas Iyer, Rajasthan Royals with Riyan Parag, Sunrisers Hyderabad captained by Pat Cummins.  
- Coaching and leadership changes have been stable with some new emerging players in key roles.  
- Auction saw high-profile purchases: Cameron Green (KKR) most expensive overseas buy; other notable uncapped expensive buys for Chennai Super Kings.  

Main Themes & Arguments from Sources:
- RCB continue as strong favorites due to defending champion status, balanced squad, and in-form key players like Virat Kohli and Rajat Patidar.  
- Punjab Kings showing consistency and topping points table with strong batting, likely playoff favorites.  
- Rajasthan Royals, with big-hitters like Vaibhav Sooryavanshi, have high-scoring capability but inconsistent results in past matches.  
- Teams with balanced bowling attacks and catching ability (e.g., Rajasthan Royals with Rabada) could sway results in crucial playoff matches.  
- Political and off-field issues (exclusion of Mustafizur Rahman) caused some controversies but didn't significantly affect main contenders' squads.
- IPL format favors teams with experience and diversified skill sets, suggesting RCB and Punjab Kings are most likely to reach final stages.  
- Emerging young talent and form fluctuations remain key unpredictable variables.  

Notable Data Points:
- Punjab Kings lead group stage with 13 points, highest net run rate among leaders.  
- RCB’s Rajat Patidar and Virat Kohli among top run-scorers, showing strong batting form.  
- Rajasthan Royals' Vaibhav Sooryavanshi leads in runs and overall points for players.  
- Auction buys like Cameron Green (KKR) expected to impact in later stages but KKR currently at bottom of the table.  
- 74-match tournament keeps intensity high with no expansion in matches this year, meaning less room for complacency.  

Conclusion:
Based on current team standings, player performances, and squad strengths, Royal Challengers Bengaluru and Punjab Kings appear strongest contenders for IPL 2026 title. Rajasthan Royals are dark horses with their explosive batting and key bowling assets. The playoff outcomes will hinge on form in knockout games and effective team strategies. RCB's defending champion experience, Punjab Kings' consistency, and Rajasthan Royals' high-impact players make them top predictions for the winner.

Sources:
- Wikipedia IPL 2026: https://en.wikipedia.org/wiki/2026_Indian_Premier_League  
- Cricbuzz IPL 2026 coverage: https://www.cricbuzz.com/cricket-series/9241/indian-premier-league-2026  
- ESPNcricinfo IPL 2026 points and analysis (summary available on Wikipedia)

### Assignment 2: Thinking Research Agent

In [70]:


# kirill vesion 
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word 'DONE:' is a control signal and not a label. Only use the word "DONE:" as per the instruction below -- it has to come at the start of a message

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that DONE: should be at the start of the final response, so it can be easily extracted and parsed.
You CANNOT and SHOULD NOT include DONE: in any part of the response except at the very beginning of the final response.

Your research brief MUST include following informatoin. However this brief should be very concice and short and not too wordy or lengthy.
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution


Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step. Emphasise more on thinking process. 
Call search and fetch tool calling as needed. Be intelligent or smart in calling the tool. Don't call tool at every iteration; Pause to think and collect data before calling tools again.
Don't call tool after every iteration. Call tool after every 2 or 3 iterations.
Not every response needs a tool call — sometimes just thinking through what you have is the right move. Wait and think more before you mark the response as "DONE:"

"""

In [72]:
# MODEL = ""
brief = run_research_agent("AI in healtcare in 2035")
display(Markdown(brief))


F50D Starting research on AI in healtcare in 2035
*******iteration: 1

[ChatCompletionMessageFunctionToolCall(id='call_VuZe0uR8xeRZYAd2zkRg4xJ3', function=Function(arguments='{"query":"AI in healthcare 2035 future trends"}', name='search_web'), type='function')]
 🔧 Calling function search_web with argument: {'query': 'AI in healthcare 2035 future trends'}
 ✅ Got results
Search Web Results: [
  {
    "title": "Artificial Intelligence and Trust to Reshape Healthcare by ...",
    "href": "https://www.about.us.hsbc.com/newsroom/press-releases/artificial-intelligence-and-trust-to-reshape-healthcare-by-2035",
    "body": "Oct 21, 2024 \u00b7 Read the highlights key barriers, opportunities, and innovations that will redefine care delivery and life sciences from the Health 2035 report."
  },
  {
    "title": "How Will AI Drive a $1 Trillion Shift in Healthcare by 2035?",
    "href": "https://healthpoint.com/business/how-will-ai-drive-a-1-trillion-shift-in-healthcare-by-2035/",
    "body": "S

Research Brief: AI in Healthcare in 2035

Key Facts and Statistics:
- AI is expected to contribute to a $1 trillion annual shift in healthcare spending by 2035, emphasizing digital-first, home-based care.
- By 2035, AI will aid 32% of diagnoses and 31% of therapeutic decisions, though only 8% of patient interactions will involve direct AI use.
- Currently, only 18% of healthcare organizations have mature AI strategies, highlighting a readiness gap.
- Administrative burden (45%) and physician reimbursement (39%) remain major barriers in delivering quality healthcare.

Main Themes and Arguments:
1. Transition to Home-Based Care: Most routine and chronic healthcare will be delivered via AI-supported wearables, implantables, and virtual command centers, shifting away from hospital-centric models.
2. AI’s Expanding Role: Applications include accelerated drug discovery, robotic surgeries, predictive analytics for early intervention, and digital twins for personalized medicine.
3. Consumer-Driven Innovation: Tech-savvy “super consumers” demand personalized, accessible, AI-enabled health solutions.
4. Challenges to Adoption: Governance, physician hesitancy, regulatory complexity, privacy, data security, and ethical concerns such as bias and equitable access must be addressed.
5. Maintaining Human Touch: Despite AI’s role in decision support, patient trust and physician involvement remain essential.
6. Economic Impact: AI-driven efficiencies are expected to reduce administrative overhead and optimize healthcare resource allocation focusing on outcome-based value.

Notable Data Points:
- AI to impact nearly one-third of diagnoses and treatments by 2035.
- Growth of in-home care supported by AI projected to surpass traditional hospital care.
- Only a minority of current healthcare providers have integrated comprehensive AI strategies.
- Data privacy frameworks like HIPAA shape AI development and deployment.

Source URLs for attribution:
- https://healthpoint.com/business/how-will-ai-drive-a-1-trillion-shift-in-healthcare-by-2035/
- https://www.about.us.hsbc.com/newsroom/press-releases/artificial-intelligence-and-trust-to-reshape-healthcare-by-2035